# 🤖 Notebook 2: Multi-Model Caption Generation

**Prerequisite:** `01_setup_and_dataset.ipynb` must be run first.  
**Reads:** `research_data/outputs/annotation_scaffold.csv`  
**Writes:** `research_data/outputs/captions_all_models.csv`

---

## Models
| Model | HuggingFace ID | VRAM needed |
|---|---|---|
| BLIP | `Salesforce/blip-image-captioning-large` | ~5 GB |
| BLIP-2 | `Salesforce/blip2-opt-2.7b` | ~15 GB (fp16: ~8 GB) |
| ViT-GPT2 | `nlpconnect/vit-gpt2-image-captioning` | ~1 GB |
| OFA | `OFA-Sys/ofa-base` | ~4 GB |

> Each model is loaded, run on all images, then **unloaded** before the next one  
> to avoid out-of-memory errors. Results are saved after each model.


In [ ]:
# CELL 1 — Install
import subprocess, sys
for pkg in ['torch','torchvision','transformers>=4.35','Pillow',
            'pandas','tqdm','numpy','requests','accelerate','einops']:
    subprocess.check_call([sys.executable,'-m','pip','install',pkg])
print('✅ Done.')

In [ ]:
# CELL 2 — Imports & config
import sys, random, time, warnings
from pathlib import Path
from typing import List, Optional

sys.path.insert(0, str(Path('.')))
import config as cfg

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import requests
from io import BytesIO
warnings.filterwarnings('ignore')

SEED   = cfg.SEED
SPLIT  = cfg.SPLIT
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU:  {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

GEN_CONFIG = cfg.GEN_CONFIG
OUTPUT_DIR = cfg.OUTPUT_DIR
FIGURES_DIR = cfg.FIGURES_DIR
print(f'Gen config: {GEN_CONFIG}')

In [ ]:
# CELL 3 — Load scaffold from NB01
scaffold_path = OUTPUT_DIR / 'annotation_scaffold.csv'
if not scaffold_path.exists():
    raise FileNotFoundError('Run 01_setup_and_dataset.ipynb first.')

df = pd.read_csv(scaffold_path)
print(f'✅ Scaffold loaded: {len(df)} images')
df[['image_id','file_name','scenario_category','primary_gt_caption']].head(3)

In [ ]:
# CELL 4 — Image loader
# Uses your local images/ folder first, then falls back to COCO CDN

def load_image(image_id: int, file_name: str) -> Image.Image:
    """
    Load a COCO image.
    Priority:
      1. images/{split}/{file_name}   ← your local folder
      2. COCO CDN URL
      3. Random noise placeholder (for debugging only)
    """
    # 1. Local file
    local = cfg.get_image_path(image_id, SPLIT, file_name)
    if local.exists():
        return Image.open(local).convert('RGB')

    # 2. COCO CDN
    url = f'http://images.cocodataset.org/{SPLIT}2014/{file_name}'
    try:
        r = requests.get(url, timeout=15)
        if r.status_code == 200:
            return Image.open(BytesIO(r.content)).convert('RGB')
    except Exception:
        pass

    # 3. Placeholder
    print(f'  ⚠️  Image not found: {local} — using noise placeholder')
    arr = (np.random.RandomState(image_id % 9999).rand(224,224,3)*255).astype(np.uint8)
    return Image.fromarray(arr)


# Quick test
row0 = df.iloc[0]
img_test = load_image(row0['image_id'], row0['file_name'])
print(f'✅ Test image: size={img_test.size}, mode={img_test.mode}')

fig, ax = plt.subplots(1,1,figsize=(4,3))
ax.imshow(img_test)
ax.set_title(f"id={row0['image_id']}\n{row0['primary_gt_caption'][:60]}", fontsize=8)
ax.axis('off')
plt.tight_layout(); plt.show()

---
## 🔧 Model Classes (Unified Interface)

In [ ]:
# CELL 5 — Base class
class CaptioningModel:
    name = 'base'
    def __init__(self, device=DEVICE):
        self.device = device
        self.model = self.processor = None
        self._loaded = False

    def load(self):
        if not self._loaded:
            print(f'  Loading {self.name}...')
            t0 = time.time()
            self._load()
            self._loaded = True
            print(f'  ✅ {self.name} ready ({time.time()-t0:.1f}s)')
        return self

    def _load(self): raise NotImplementedError

    def generate_caption(self, image: Image.Image) -> str:
        raise NotImplementedError

    def generate_batch(self, images: List[Image.Image], desc='') -> List[str]:
        captions = []
        for img in tqdm(images, desc=desc or self.name):
            try:
                captions.append(self.generate_caption(img))
            except Exception as e:
                print(f'  ⚠️  {self.name} error: {e}')
                captions.append('[ERROR]')
        return captions

    def unload(self):
        self.model = self.processor = None
        if DEVICE == 'cuda': torch.cuda.empty_cache()
        self._loaded = False
        print(f'  🗑  {self.name} unloaded')

print('✅ Base class ready.')

In [ ]:
# CELL 6 — BLIP
# Also provides generate_with_attention() used by NB05
from transformers import BlipProcessor, BlipForConditionalGeneration

class BLIPModel(CaptioningModel):
    name  = 'blip'
    HF_ID = cfg.MODEL_HF_IDS['blip']

    def _load(self):
        self.processor = BlipProcessor.from_pretrained(self.HF_ID)
        self.model = BlipForConditionalGeneration.from_pretrained(
            self.HF_ID,
            torch_dtype=torch.float16 if self.device=='cuda' else torch.float32
        ).to(self.device).eval()

    def generate_caption(self, image: Image.Image) -> str:
        inputs = self.processor(image, return_tensors='pt').to(self.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, **GEN_CONFIG)
        return self.processor.decode(out[0], skip_special_tokens=True).strip()

    def generate_with_attention(self, image: Image.Image):
        """
        Returns (caption, token_attentions).
        token_attentions: list of np.ndarray shape (SPATIAL_RES^2,) per generated token.
        Used by NB05 for JS/KL divergence analysis.
        """
        SRES = cfg.SPATIAL_RES
        inputs = self.processor(image, return_tensors='pt').to(self.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs, **GEN_CONFIG,
                output_attentions=True,
                return_dict_in_generate=True
            )
        caption = self.processor.decode(out.sequences[0], skip_special_tokens=True).strip()
        token_attentions = []
        if hasattr(out, 'cross_attentions') and out.cross_attentions:
            for step_attn in out.cross_attentions:
                last = step_attn[-1]          # last decoder layer
                attn = last[0].mean(dim=0).squeeze()  # avg over heads
                n_p = attn.shape[-1]
                gs  = int(n_p**0.5)
                attn_grid = attn[:gs*gs].cpu().float().numpy().reshape(gs, gs)
                attn_resized = np.array(
                    Image.fromarray(attn_grid).resize((SRES,SRES), Image.BILINEAR)
                ).flatten().astype(np.float64)
                attn_resized += 1e-10
                attn_resized /= attn_resized.sum()
                token_attentions.append(attn_resized)
        return caption, token_attentions

print('✅ BLIPModel defined.')

In [ ]:
# CELL 7 — BLIP-2
from transformers import Blip2Processor, Blip2ForConditionalGeneration

class BLIP2Model(CaptioningModel):
    name  = 'blip2'
    HF_ID = cfg.MODEL_HF_IDS['blip2']

    def _load(self):
        self.processor = Blip2Processor.from_pretrained(self.HF_ID)
        dtype = torch.float16 if self.device=='cuda' else torch.float32
        self.model = Blip2ForConditionalGeneration.from_pretrained(
            self.HF_ID, torch_dtype=dtype,
            device_map='auto' if self.device=='cuda' else None
        ).eval()
        if self.device != 'cuda':
            self.model = self.model.to(self.device)

    def generate_caption(self, image: Image.Image) -> str:
        dtype = torch.float16 if self.device=='cuda' else torch.float32
        inputs = self.processor(image, return_tensors='pt').to(self.device, dtype)
        with torch.no_grad():
            out = self.model.generate(**inputs, **GEN_CONFIG)
        return self.processor.decode(out[0], skip_special_tokens=True).strip()

print('✅ BLIP2Model defined.')

In [ ]:
# CELL 8 — ViT-GPT2
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer

class ViTGPT2Model(CaptioningModel):
    name  = 'vit_gpt2'
    HF_ID = cfg.MODEL_HF_IDS['vit_gpt2']

    def _load(self):
        self.feature_extractor = ViTImageProcessor.from_pretrained(self.HF_ID)
        self.tokenizer = AutoTokenizer.from_pretrained(self.HF_ID)
        self.model = VisionEncoderDecoderModel.from_pretrained(
            self.HF_ID).to(self.device).eval()

    def generate_caption(self, image: Image.Image) -> str:
        if image.mode != 'RGB': image = image.convert('RGB')
        pv = self.feature_extractor(image, return_tensors='pt').pixel_values.to(self.device)
        with torch.no_grad():
            out = self.model.generate(
                pv,
                max_length=GEN_CONFIG['max_new_tokens'],
                num_beams=GEN_CONFIG['num_beams'],
                early_stopping=GEN_CONFIG['early_stopping']
            )
        return self.tokenizer.decode(out[0], skip_special_tokens=True).strip()

print('✅ ViTGPT2Model defined.')

In [ ]:
# CELL 9 — OFA
class OFAModel(CaptioningModel):
    name  = 'ofa'
    HF_ID = cfg.MODEL_HF_IDS['ofa']

    def _load(self):
        try:
            from transformers import OFATokenizer, OFAModel as _OFA
            from torchvision import transforms
            self.tokenizer = OFATokenizer.from_pretrained(self.HF_ID)
            self.model = _OFA.from_pretrained(self.HF_ID, use_cache=False).to(self.device).eval()
            mean, std = [0.5]*3, [0.5]*3
            self.transform = transforms.Compose([
                transforms.Resize((480,480),
                    interpolation=transforms.InterpolationMode.BICUBIC),
                transforms.ToTensor(),
                transforms.Normalize(mean,std)
            ])
            self._ok = True
        except Exception as e:
            print(f'  ⚠️  OFA unavailable: {e}')
            self._ok = False

    def generate_caption(self, image: Image.Image) -> str:
        if not self._ok: return '[OFA_UNAVAILABLE]'
        if image.mode != 'RGB': image = image.convert('RGB')
        patch = self.transform(image).unsqueeze(0).to(self.device)
        inputs = self.tokenizer(
            [' what does the image describe?'],
            return_tensors='pt', padding=True
        ).to(self.device)
        with torch.no_grad():
            out = self.model.generate(
                input_ids=inputs.input_ids,
                patch_images=patch,
                num_beams=GEN_CONFIG['num_beams'],
                max_length=GEN_CONFIG['max_new_tokens']+10,
                early_stopping=GEN_CONFIG['early_stopping']
            )
        return self.tokenizer.decode(out[0], skip_special_tokens=True).strip()

print('✅ OFAModel defined.')

---
## 🔄 Generation Loop

Each model is loaded → run → saved → unloaded before the next.  
A checkpoint CSV is written after **each model** — safe to restart if interrupted.


In [ ]:
# CELL 10 — Load checkpoint if resuming
checkpoint_path = OUTPUT_DIR / 'captions_checkpoint.csv'
if checkpoint_path.exists():
    df_check = pd.read_csv(checkpoint_path)
    # Restore any already-computed caption columns
    for col in ['blip_caption','blip2_caption','ofa_caption','vit_gpt2_caption']:
        if col in df_check.columns:
            df[col] = df_check[col]
    print(f'♻️  Checkpoint loaded. Filled columns:')
    for col in ['blip_caption','blip2_caption','ofa_caption','vit_gpt2_caption']:
        n = df[col].notna().sum() if col in df.columns else 0
        print(f'   {col}: {n}/{len(df)}')
else:
    print('No checkpoint found. Starting fresh.')

In [ ]:
# CELL 11 — Run BLIP
# Skip if already done
if df['blip_caption'].notna().sum() == len(df):
    print('⏭  BLIP already done — skipping.')
else:
    blip = BLIPModel(); blip.load()
    caps = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='BLIP'):
        img = load_image(row['image_id'], row['file_name'])
        caps.append(blip.generate_caption(img))
    df['blip_caption'] = caps
    df.to_csv(checkpoint_path, index=False)
    print(f'✅ BLIP done. Sample: {caps[0]}')
    blip.unload()

In [ ]:
# CELL 12 — Run BLIP-2
if df['blip2_caption'].notna().sum() == len(df):
    print('⏭  BLIP-2 already done — skipping.')
else:
    blip2 = BLIP2Model(); blip2.load()
    caps = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='BLIP-2'):
        img = load_image(row['image_id'], row['file_name'])
        caps.append(blip2.generate_caption(img))
    df['blip2_caption'] = caps
    df.to_csv(checkpoint_path, index=False)
    print(f'✅ BLIP-2 done. Sample: {caps[0]}')
    blip2.unload()

In [ ]:
# CELL 13 — Run ViT-GPT2
if df['vit_gpt2_caption'].notna().sum() == len(df):
    print('⏭  ViT-GPT2 already done — skipping.')
else:
    vgpt2 = ViTGPT2Model(); vgpt2.load()
    caps = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='ViT-GPT2'):
        img = load_image(row['image_id'], row['file_name'])
        caps.append(vgpt2.generate_caption(img))
    df['vit_gpt2_caption'] = caps
    df.to_csv(checkpoint_path, index=False)
    print(f'✅ ViT-GPT2 done. Sample: {caps[0]}')
    vgpt2.unload()

In [ ]:
# CELL 14 — Run OFA
if df['ofa_caption'].notna().sum() == len(df):
    print('⏭  OFA already done — skipping.')
else:
    ofa = OFAModel(); ofa.load()
    caps = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='OFA'):
        img = load_image(row['image_id'], row['file_name'])
        caps.append(ofa.generate_caption(img))
    df['ofa_caption'] = caps
    df.to_csv(checkpoint_path, index=False)
    print(f'✅ OFA done. Sample: {caps[0]}')
    ofa.unload()

In [ ]:
ofa_caption = None

In [ ]:
# CELL 15 — Side-by-side comparison table
MODEL_COLS = {
    'BLIP':     'blip_caption',
    'BLIP-2':   'blip2_caption',
    'ViT-GPT2': 'vit_gpt2_caption',
    #'OFA':      'ofa_caption',
}
print('=' * 90)
print('CAPTION COMPARISON — FIRST 5 IMAGES')
print('=' * 90)
for _, row in df.head(5).iterrows():
    print(f'\n📷 id={row["image_id"]}  [{row["scenario_category"]}]')
    print(f'   GT:         {str(row["primary_gt_caption"])[:90]}')
    for mname, col in MODEL_COLS.items():
        val = str(row[col]) if col in row and pd.notna(row[col]) else '[not run]'
        print(f'   {mname:<10}: {val[:90]}')

In [ ]:
# CELL 16 — Caption length distribution
import seaborn as sns
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Caption Length Distribution per Model', fontsize=13, fontweight='bold')

lengths = {}
for mname, col in MODEL_COLS.items():
    if col in df.columns:
        lengths[mname] = df[col].dropna().apply(lambda x: len(str(x).split())).values

axes[0].boxplot(list(lengths.values()), labels=list(lengths.keys()), patch_artist=True)
axes[0].set_ylabel('Words'); axes[0].set_title('Box Plot'); axes[0].grid(axis='y', alpha=0.3)

for mname, lens in lengths.items():
    sns.kdeplot(lens, ax=axes[1], label=mname, fill=True, alpha=0.25)
axes[1].set_xlabel('Caption length (words)'); axes[1].set_title('KDE')
axes[1].legend()

plt.tight_layout()
fig.savefig(FIGURES_DIR/'caption_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# CELL 17 — Save final caption DataFrame
out_path = OUTPUT_DIR / 'captions_all_models.csv'
df.to_csv(out_path, index=False)

print(f'💾 captions_all_models.csv saved: {out_path}')
print('\nFill rates:')
for mname, col in MODEL_COLS.items():
    n = df[col].notna().sum() if col in df.columns else 0
    print(f'  {mname:<12}: {n}/{len(df)} ({n/len(df)*100:.0f}%)')

print('\n✅ Notebook 2 COMPLETE — run 03_metric_computation.ipynb next')